# Historical crop-label scratchpad

This preserved notebook is not a validated Run All recipe. The extraction loop stores `output["coords"]`, which are already pixel coordinates. Later cells multiply x/y/w/h by image dimensions as though they were normalized. Running both sections produces out-of-bounds labels on ordinary images. Resolve which coordinate representation was intended before generating crop-training CSVs. Original code below is preserved unchanged as historical evidence; data and outputs are not included.


In [ ]:
import cv2
import glob
import os
import numpy as np
import pandas as pd

from tqdm import tqdm

In [ ]:
def crop_roi(img):
    img = img[5:-5, 5:-5]

    output = cv2.connectedComponentsWithStats((img > 10).astype(np.uint8)[:, :], 8, cv2.CV_32S) #
    stats = output[2]

    idx = stats[1:, 4].argmax() + 1
    x1, y1, w, h = stats[idx][:4]
    x1 = max(0, x1 - 5)
    y1 = max(0, y1 - 5)
    
    img_h, img_w = img.shape[:2]

    return {"coords": (x1, y1, w, h), "normalized_coords": (x1 / img_w, y1 / img_h, w / img_w, h / img_h), "img": img[y1:y1 + h, x1:x1 + w]}

In [ ]:
ROOT = "/mnt/stor/datasets/kaggle/rsna-breast-cancer-detection/"

images = glob.glob(os.path.join(ROOT, "train_png/*/*.png"))

In [ ]:
images[:2]

In [ ]:
cols = ["filename", "x", "y", "w", "h"]
df_dict = {c: [] for c in cols}
for im in tqdm(images):
    img = cv2.imread(im, cv2.IMREAD_GRAYSCALE)
    output = crop_roi(img)
    df_dict["filename"].append("/".join(im.split("/")[-2:]))
    df_dict["x"].append(output["coords"][0])
    df_dict["y"].append(output["coords"][1])
    df_dict["w"].append(output["coords"][2])
    df_dict["h"].append(output["coords"][3])

In [ ]:
coords_df = pd.DataFrame(df_dict)

In [ ]:
from collections import defaultdict
from PIL import Image 

image_sizes = defaultdict(list)

for row_idx, row in tqdm(coords_df.iterrows(), total=len(coords_df)):
    img_size = Image.open(os.path.join(ROOT, "train_png", row.filename)).size
    image_sizes["filename"].append(row.filename)
    image_sizes["img_w"].append(img_size[0])
    image_sizes["img_h"].append(img_size[1])

In [ ]:
image_df = pd.DataFrame(image_sizes)
coords_df["x"] = (coords_df.x * image_df.img_w).astype("int")
coords_df["y"] = (coords_df.y * image_df.img_h).astype("int")
coords_df["w"] = (coords_df.w * image_df.img_w).astype("int")
coords_df["h"] = (coords_df.h * image_df.img_h).astype("int")
coords_df

In [ ]:
coords_df[["filename", "x", "y", "w", "h"]].to_csv(os.path.join(ROOT, "crop_coords_using_connected_components.csv"), index=False)

In [ ]:
import matplotlib.pyplot as plt

idx = np.random.choice(len(coords_df))
row = coords_df.iloc[idx]

img = cv2.imread(os.path.join(ROOT, "train_png", row.filename), cv2.IMREAD_GRAYSCALE)
img_crop = img[row.y:row.y + row.h, row.x:row.x + row.w]

plt.subplot(1, 2, 1)
plt.imshow(img, cmap="gray")
plt.subplot(1, 2, 2)
plt.imshow(img_crop, cmap="gray")
plt.show()

In [ ]:
coords_df["image_id"] = coords_df.filename.apply(lambda x: x.split("/")[-1].replace(".png", "")).astype("int")
coords_df

In [ ]:
from skp.toolbox.functions import create_double_cv


train_df = pd.read_csv(os.path.join(ROOT, "train.csv"))
coords_df = coords_df.merge(train_df[["patient_id", "image_id"]], on="image_id")
coords_df = create_double_cv(coords_df, "patient_id", 10, 10)
coords_df

In [ ]:
coords_df.to_csv(os.path.join(ROOT, "train_coords_kfold.csv"), index=False)

In [ ]:
coords_df